# Frequency Model — M3 Final Implementation

Poisson GLM with **24 features**:
- 5 coverage type dummies
- 8 temporal / lag features (lag_1m, rolling_3m, lag_12m, time_index, policy_year, month_cos, is_winter, avg_prior_claims)
- 4 external features (wind speed, unemployment, GDP, storm flag)
- 1 region dummy (South East)
- 6 interaction terms (lag×trend, rolling×storm, gdp×trend, unemp×trend, lag1×winter, lag12×trend)

**Split**: sklearn train_test_split, random 80/10/10, seed=42  
**Metrics**: Poisson deviance, calibration%, Gini (count), MAE, RMSE, mean actual, mean predicted, total actual, total predicted

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

SEED = 42
OUT  = '../output/'
RAW  = r'C:\Users\DorothyCheruiyot\Desktop\Projects\insurance_claims\data\insurance_data_features.xlsx'


## Helper Functions — Metrics & Utilities

In [2]:
def poisson_deviance(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.clip(np.array(y_pred, dtype=float), 1e-10, None)
    term   = np.where(y_true > 0, y_true * np.log(y_true / y_pred), 0.0)
    return float(2 * np.sum(term - (y_true - y_pred)))

def calibration_pct(y_true, y_pred):
    return float((np.sum(y_pred) - np.sum(y_true)) / np.sum(y_true) * 100)

def gini_coefficient(y_true, y_pred):
    """Count-based Gini — ranks segments by predicted claim count."""
    idx = np.argsort(y_pred)
    ys  = np.array(y_true, dtype=float)[idx]
    cs  = np.cumsum(ys)
    if cs[-1] == 0: return 0.0
    n   = len(ys)
    return float((n + 1 - 2 * np.sum(cs) / cs[-1]) / n)

def lift_table(y_true, y_pred, n_bins=10):
    df_l = pd.DataFrame({'y': np.array(y_true, float),
                         'p': np.array(y_pred, float)})
    df_l['decile'] = pd.qcut(df_l['p'], n_bins, labels=False, duplicates='drop')
    return (df_l.groupby('decile')
            .agg(actual_mean=('y','mean'), pred_mean=('p','mean'),
                 count=('y','count'))
            .reset_index())

def score(name, y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    return {
        'model'           : name,
        'poisson_deviance': round(poisson_deviance(y_true, y_pred), 4),
        'calibration_%'   : round(calibration_pct(y_true, y_pred), 3),
        'gini'            : round(gini_coefficient(y_true, y_pred), 4),
        'mae'             : round(float(mean_absolute_error(y_true, y_pred)), 4),
        'rmse'            : round(float(np.sqrt(np.mean((y_true - y_pred)**2))), 4),
        'mean_actual'     : round(float(np.mean(y_true)), 4),
        'mean_predicted'  : round(float(np.mean(y_pred)), 4),
        'total_actual'    : int(np.sum(y_true)),
        'total_predicted' : round(float(np.sum(y_pred)), 1),
    }

METRIC_KEYS = ['poisson_deviance','calibration_%','gini','mae','rmse',
               'mean_actual','mean_predicted','total_actual','total_predicted']
METRIC_LBLS = ['Poisson Deviance','Calibration %','Gini (count)','MAE','RMSE',
               'Mean Actual','Mean Predicted','Total Actual','Total Predicted']

In [3]:
sheets    = pd.read_excel(RAW, sheet_name=None, dtype=str)
customers = sheets['Customers'].copy()
policies  = sheets['Policies'].copy()
claims    = sheets['Claims'].copy()
external  = sheets['External'].copy()

for c in ['Age','Credit Score','Prior Claims Count',
          'Tenure Years','Late Payments 12M']:
    customers[c] = pd.to_numeric(customers[c], errors='coerce')
customers['income_high'] = customers['Income Band'].isin(
    ['>100k','60-100k']).astype(float)

policies['Start Date'] = pd.to_datetime(policies['Start Date'], errors='coerce')
policies['End Date']   = pd.to_datetime(policies['End Date'],   errors='coerce')
claims['Claim Date']   = pd.to_datetime(claims['Claim Date'],   errors='coerce')
claims['year_month']   = claims['Claim Date'].dt.to_period('M').astype(str)

for c in ['Avg Rainfall Mm','Avg Wind Speed Kmh','Flood Risk Index',
          'Gdp Growth Rate','Cpi Inflation','Unemployment Rate',
          'Avg Property Price Gbp','Storm Event Flag']:
    external[c] = pd.to_numeric(external[c], errors='coerce')


In [4]:
# build segment table
def get_active_months(row):
    try:
        return [str(m) for m in
                pd.period_range(row['Start Date'], row['End Date'], freq='M')]
    except Exception:
        return []


policies['_months'] = policies.apply(get_active_months, axis=1)
pol_exp = (policies[['Policy Id','Customer Id','Coverage Type','_months']]
           .explode('_months').rename(columns={'_months':'year_month'}))
pol_exp = pol_exp.merge(
    customers[['Customer Id','Region','Credit Score','Prior Claims Count',
               'Age','Tenure Years','income_high']],
    on='Customer Id', how='left')

OBS_START, OBS_END = '2021-01', '2024-12'
pol_exp = pol_exp[(pol_exp['year_month'] >= OBS_START) &
                  (pol_exp['year_month'] <= OBS_END)].copy()

seg_exposure = (pol_exp
                .groupby(['Coverage Type','Region','year_month'])
                .size()
                .reset_index(name='earned_exposure'))

seg_customer = (pol_exp
                .groupby(['Coverage Type','Region','year_month'])
                .agg(avg_credit_score=('Credit Score','mean'),
                     avg_prior_claims=('Prior Claims Count','median'),
                     avg_age =('Age','mean'),
                     avg_tenure=('Tenure Years','median'),
                     pct_high_income =('income_high','mean'))
                .reset_index())

claims_j = claims.merge(
    policies[['Policy Id','Customer Id','Coverage Type']],
    on='Policy Id', how='left', suffixes=('_cl','_pol'))
claims_j = claims_j.rename(columns={'Customer Id_cl':'Customer Id'})
claims_j = claims_j.merge(
    customers[['Customer Id','Region']], on='Customer Id', how='left')
claims_j['Coverage Type'] = claims_j['Coverage Type_cl']

seg_claims = (claims_j
              .groupby(['Coverage Type','Region','year_month'])
              .size()
              .reset_index(name='claim_count'))

freq_df = (seg_exposure
           .merge(seg_claims,   on=['Coverage Type','Region','year_month'], how='left')
           .merge(seg_customer, on=['Coverage Type','Region','year_month'], how='left'))
freq_df['claim_count'] = freq_df['claim_count'].fillna(0).astype(int)

ext = external.rename(columns={'Year Month':'year_month'})
freq_df = freq_df.merge(
    ext[['Region','year_month','Avg Rainfall Mm','Avg Wind Speed Kmh',
         'Flood Risk Index','Gdp Growth Rate','Cpi Inflation',
         'Unemployment Rate','Avg Property Price Gbp','Storm Event Flag']],
    on=['Region','year_month'], how='left')


freq_df.head()

,Coverage Type,Region,year_month,earned_exposure,claim_count,avg_credit_score,avg_prior_claims,avg_age,avg_tenure,pct_high_income,Avg Rainfall Mm,Avg Wind Speed Kmh,Flood Risk Index,Gdp Growth Rate,Cpi Inflation,Unemployment Rate,Avg Property Price Gbp,Storm Event Flag
0,Commercial,East Anglia,2021-01,9,0,634.222222,1.0,47.000000,10.0,0.444444,35.5,5.6,2.36,0.0162,0.0213,0.0616,165500,0
1,Commercial,East Anglia,2021-02,16,1,643.375000,1.0,50.125000,8.5,0.375000,30.4,11.5,1.83,0.0115,0.0631,0.0539,328500,0
2,Commercial,East Anglia,2021-03,22,0,638.818182,1.0,50.454545,8.5,0.409091,56.2,9.3,2.73,0.0126,0.0351,0.0447,270400,0
3,Commercial,East Anglia,2021-04,30,2,630.533333,1.0,50.766667,8.0,0.400000,52.5,21.2,3.06,0.0132,0.0501,0.0318,227600,0
4,Commercial,East Anglia,2021-05,34,0,636.382353,1.0,50.000000,6.5,0.382353,62.6,19.6,3.00,0.0320,0.0438,0.0436,257400,1


In [5]:
# feature engineering
freq_df = freq_df.sort_values(
    ['Coverage Type','Region','year_month']).reset_index(drop=True)

period_idx = pd.PeriodIndex(freq_df['year_month'], freq='M')

freq_df['time_index'] = period_idx.astype(int) - period_idx.astype(int).min()
freq_df['policy_year'] = period_idx.year - 2021
freq_df['month_num']= period_idx.month
freq_df['month_cos']= np.cos(2 * np.pi * freq_df['month_num'] / 12)
freq_df['month_sin']= np.sin(2 * np.pi * freq_df['month_num'] / 12)
freq_df['is_winter']= freq_df['month_num'].isin([10,11,12,1,2,3]).astype(int)
freq_df['log_exposure']= np.log(freq_df['earned_exposure'].clip(lower=1))
freq_df['flood_x_rain']= freq_df['Flood Risk Index'] * freq_df['Avg Rainfall Mm']

grp = freq_df.groupby(['Coverage Type','Region'])['claim_count']
freq_df['lag_1m']= grp.shift(1)
freq_df['lag_12m'] = grp.shift(12)
freq_df['rolling_3m'] = grp.shift(1).rolling(3, min_periods=1).mean()

for col in ['lag_1m','lag_12m','rolling_3m']:
    seg_mean = freq_df.groupby(['Coverage Type','Region'])[col].transform('mean')
    freq_df[col] = freq_df[col].fillna(seg_mean)

cov_dummies = pd.get_dummies(freq_df['Coverage Type'], prefix='cov', drop_first=True, dtype=int)
reg_dummies = pd.get_dummies(freq_df['Region'],        prefix='reg', drop_first=True, dtype=int)
freq_df = pd.concat([freq_df, cov_dummies, reg_dummies], axis=1)

COV_DUMMIES = list(cov_dummies.columns)


In [6]:
# split
df_trainval, df_test = train_test_split(freq_df, test_size=0.10, random_state=SEED)
df_train, df_val    = train_test_split(df_trainval, test_size=0.1111, random_state=SEED)

df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

TARGET = 'claim_count'
OFFSET = 'log_exposure'
EXP    = 'earned_exposure'

In [7]:

SCALE_COLS = ['avg_credit_score','avg_prior_claims','avg_age',
              'avg_tenure','pct_high_income',
              'Avg Rainfall Mm','Avg Wind Speed Kmh','Flood Risk Index',
              'Gdp Growth Rate','Cpi Inflation','Unemployment Rate',
              'Avg Property Price Gbp','flood_x_rain',
              'lag_1m','lag_12m','rolling_3m',
              'time_index','policy_year']
SCALE_COLS = [c for c in SCALE_COLS if c in df_train.columns]

scaler = StandardScaler()
scaler.fit(df_train[SCALE_COLS])

def apply_scaler(df_split):
    vals = scaler.transform(df_split[SCALE_COLS])
    for i, c in enumerate(SCALE_COLS):
        df_split[f'{c}_sc'] = vals[:, i]
    return df_split

df_train = apply_scaler(df_train)
df_val   = apply_scaler(df_val)
df_test  = apply_scaler(df_test)

df_train.head()

,Coverage Type,Region,year_month,earned_exposure,claim_count,avg_credit_score,avg_prior_claims,avg_age,avg_tenure,pct_high_income,...,Gdp Growth Rate_sc,Cpi Inflation_sc,Unemployment Rate_sc,Avg Property Price Gbp_sc,flood_x_rain_sc,lag_1m_sc,lag_12m_sc,rolling_3m_sc,time_index_sc,policy_year_sc
0,Commercial,North East,2023-12,114,3,647.745614,1.0,45.429825,7.0,0.324561,...,-1.088679,1.274537,-0.507854,0.177411,-1.271844,-0.484799,-0.497074,0.649694,0.838051,0.457444
1,Commercial,South West,2024-12,57,3,643.298246,1.0,44.017544,7.0,0.403509,...,-1.814972,-0.463278,-1.811889,-0.085296,-0.127927,-0.484799,-0.497074,-0.748106,1.699470,1.346105
2,Motor,London,2023-01,79,4,653.468354,1.0,49.063291,6.0,0.379747,...,-0.437520,-2.351923,-0.458459,0.129018,-1.097085,-0.484799,0.316817,0.999144,0.048417,0.457444
3,Health,Scotland,2021-09,58,1,641.431034,1.0,49.051724,8.5,0.482759,...,-1.464348,-0.043579,-0.735072,-0.277141,-0.505060,0.218069,0.226385,-0.049206,-1.100141,-1.319877
4,Travel,London,2021-04,28,1,644.535714,1.0,50.178571,8.0,0.357143,...,0.075895,-0.666570,1.892757,-0.526022,-0.723393,-1.187668,-0.203169,-1.796455,-1.459066,-1.319877


In [8]:
for df_split in [df_train, df_val, df_test]:
    df_split['lag1_x_trend'] = df_split['lag_1m_sc']* df_split['time_index_sc']
    df_split['lag12_x_trend'] = df_split['lag_12m_sc'] * df_split['time_index_sc']
    df_split['rolling3_x_storm'] = df_split['rolling_3m_sc'] * df_split['Storm Event Flag']
    df_split['gdp_x_trend'] = df_split['Gdp Growth Rate_sc'] * df_split['time_index_sc']
    df_split['unemp_x_trend'] = df_split['Unemployment Rate_sc'] * df_split['time_index_sc']
    df_split['lag1_x_winter']= df_split['lag_1m_sc']* df_split['is_winter']

INTERACTION_FEATS = ['lag1_x_trend','lag12_x_trend','rolling3_x_storm',
                     'gdp_x_trend','unemp_x_trend','lag1_x_winter']


In [9]:
COV_DUMMIES
REG_DUMMIES = list(reg_dummies.columns)
REG_DUMMIES

['reg_London',
 'reg_Midlands',
 'reg_North East',
 'reg_North West',
 'reg_Scotland',
 'reg_South East',
 'reg_South West',
 'reg_Wales',
 'reg_Yorkshire']

In [10]:
M3_FEATURES = list(dict.fromkeys([f for f in
    COV_DUMMIES
    + ['is_winter','avg_prior_claims_sc','lag_1m_sc','rolling_3m_sc',
       'lag_12m_sc','time_index_sc','policy_year_sc','month_cos']
    + ['Storm Event Flag','Avg Wind Speed Kmh_sc',
       'Unemployment Rate_sc','Gdp Growth Rate_sc']
    + REG_DUMMIES
    + INTERACTION_FEATS
    if f in df_train.columns]))

len(M3_FEATURES), M3_FEATURES

(32,
 ['cov_Health',
  'cov_Home',
  'cov_Life',
  'cov_Motor',
  'cov_Travel',
  'is_winter',
  'avg_prior_claims_sc',
  'lag_1m_sc',
  'rolling_3m_sc',
  'lag_12m_sc',
  'time_index_sc',
  'policy_year_sc',
  'month_cos',
  'Storm Event Flag',
  'Avg Wind Speed Kmh_sc',
  'Unemployment Rate_sc',
  'Gdp Growth Rate_sc',
  'reg_London',
  'reg_Midlands',
  'reg_North East',
  'reg_North West',
  'reg_Scotland',
  'reg_South East',
  'reg_South West',
  'reg_Wales',
  'reg_Yorkshire',
  'lag1_x_trend',
  'lag12_x_trend',
  'rolling3_x_storm',
  'gdp_x_trend',
  'unemp_x_trend',
  'lag1_x_winter'])

## Step 8 — Fit Poisson GLM

In [11]:
Xc_train = sm.add_constant(df_train[M3_FEATURES].fillna(0), has_constant='add')

glm = sm.GLM(
    df_train[TARGET],
    Xc_train,
    family   = sm.families.Poisson(link=sm.families.links.Log()),
    exposure = np.exp(df_train[OFFSET])
)
result = glm.fit(maxiter=300, disp=False)


result.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:            claim_count   No. Observations:                 2304
Model:                            GLM   Df Residuals:                     2271
Model Family:                 Poisson   Df Model:                           32
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -3640.5
Date:                Wed, 03 Jun 2026   Deviance:                       2634.9
Time:                        08:58:34   Pearson chi2:                 2.40e+03
No. Iterations:                     5   Pseudo R-squ. (CS):            0.01464
Covariance Type:            nonrobust                                         
=========================================================================================
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const                    -3.8719      0.072    -53.814      0.000      -4.013      -3.731
cov_Health                0.0024      0.055      0.043      0.966      -0.105       0.110
cov_Home                  0.0192      0.055      0.349      0.727      -0.089       0.127
cov_Life                  0.0022      0.056      0.040      0.968      -0.107       0.111
cov_Motor                 0.0740      0.054      1.375      0.169      -0.032       0.180
cov_Travel               -0.0630      0.056     -1.129      0.259      -0.172       0.046
is_winter                 0.0050      0.074      0.068      0.946      -0.140       0.150
avg_prior_claims_sc       0.0372      0.028      1.329      0.184      -0.018       0.092
lag_1m_sc                 0.0023      0.026      0.091      0.928      -0.048       0.053
rolling_3m_sc            -0.0490      0.023     -2.150      0.032      -0.094      -0.004
lag_12m_sc               -0.0374      0.020     -1.891      0.059      -0.076       0.001
time_index_sc             0.2018      0.076      2.662      0.008       0.053       0.350
policy_year_sc           -0.1136      0.072     -1.581      0.114      -0.254       0.027
month_cos                -0.0022      0.057     -0.039      0.969      -0.114       0.110
Storm Event Flag         -0.0639      0.056     -1.131      0.258      -0.175       0.047
Avg Wind Speed Kmh_sc     0.0091      0.022      0.422      0.673      -0.033       0.052
Unemployment Rate_sc     -0.0002      0.018     -0.013      0.989      -0.035       0.034
Gdp Growth Rate_sc        0.0093      0.017      0.563      0.573      -0.023       0.042
reg_London               -0.0376      0.073     -0.514      0.607      -0.181       0.106
reg_Midlands              0.0278      0.071      0.393      0.694      -0.111       0.166
reg_North East           -0.0305      0.071     -0.431      0.667      -0.169       0.108
reg_North West           -0.0336      0.072     -0.466      0.641      -0.175       0.108
reg_Scotland              0.0277      0.071      0.392      0.695      -0.111       0.166
reg_South East            0.0714      0.069      1.027      0.304      -0.065       0.208
reg_South West            0.0383      0.072      0.534      0.594      -0.102       0.179
reg_Wales                -0.0043      0.071     -0.061      0.951      -0.143       0.134
reg_Yorkshire            -0.0066      0.071     -0.093      0.926      -0.145       0.132
lag1_x_trend             -0.0133      0.019     -0.703      0.482      -0.051       0.024
lag12_x_trend             0.0156      0.021      0.729      0.466      -0.026       0.058
rolling3_x_storm          0.0096      0.058      0.167      0.868      -0.103       0.123
gdp_x_trend               0.0065      0.018      0.362      0.718      -0.029       0.042
unemp_x_trend          

## Step 9 — Overdispersion Check

In [12]:
mu_train   = result.fittedvalues.values
y_train    = df_train[TARGET].values
dispersion = result.pearson_chi2 / result.df_resid
dispersion


np.float64(1.0568758949849784)

In [13]:
def predict_m3(df_split):
    Xc = sm.add_constant(df_split[M3_FEATURES].fillna(0), has_constant='add')
    for c in set(result.params.index) - set(Xc.columns):
        Xc[c] = 0
        
    return result.predict(np.array(Xc[result.params.index]),
                          exposure=np.exp(np.array(df_split[OFFSET])))

pred_train = predict_m3(df_train)
pred_val   = predict_m3(df_val)
pred_test  = predict_m3(df_test)

s_train = score('M3 — Train', df_train[TARGET], pred_train)
s_val   = score('M3 — Val',   df_val[TARGET],   pred_val)
s_test  = score('M3 — Test',  df_test[TARGET],  pred_test)

results_df = pd.DataFrame([s_train, s_val, s_test]).set_index('model')
results_df

,poisson_deviance,calibration_%,gini,mae,rmse,mean_actual,mean_predicted,total_actual,total_predicted
model,,,,,,,,,
M3 — Train,2634.8880,-0.000,0.1814,1.0469,1.3584,1.7396,1.7396,4008,4008.0
M3 — Val,282.3654,4.267,0.1824,0.9574,1.2281,1.7222,1.7957,496,517.2
M3 — Test,350.9911,6.002,0.1461,1.0771,1.3940,1.7222,1.8256,496,525.8


In [14]:
coef_df = pd.DataFrame({
    'feature' : result.params.index,
    'coef': result.params.values,
    'std_err': result.bse.values,
    'z_stat': result.tvalues.values,
    'p_value' : result.pvalues.values,
    'IRR': np.exp(result.params.values),
    'IRR_lo95': np.exp(result.conf_int()[0].values),
    'IRR_hi95': np.exp(result.conf_int()[1].values),
}).query("feature != 'const'").sort_values('p_value').reset_index(drop=True)

coef_df['sig'] = coef_df['p_value'].apply(
    lambda p: '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else '')

coef_df.to_csv(f'{OUT}freq_m3_coefficients.csv', index=False)

coef_df[['feature','IRR','IRR_lo95','IRR_hi95','p_value','sig']]

,feature,IRR,IRR_lo95,IRR_hi95,p_value,sig
0,time_index_sc,NaN,1.054652,1.419759,0.007777,**
1,rolling_3m_sc,NaN,0.910624,0.995681,0.031558,*
2,lag_12m_sc,NaN,0.926681,1.001356,0.058565,
3,policy_year_sc,NaN,0.775313,1.027649,0.113987,
4,cov_Motor,NaN,0.968973,1.196718,0.169223,
5,avg_prior_claims_sc,NaN,0.982462,1.096568,0.183928,
6,Storm Event Flag,NaN,0.839786,1.047927,0.257972,
7,cov_Travel,NaN,0.841635,1.047476,0.258918,
8,reg_South East,NaN,0.937257,1.230666,0.304257,
9,lag1_x_winter,NaN,0.965516,1.094235,0.389358,
